<a href="https://colab.research.google.com/github/MonaKhadija/Data-Analytics-and-Artificial-Intelligence-Project-DAAI/blob/main/Project3_Khadija_bassiouni.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Project 3: Machine Learning for Predicting Trading Signals**

*Student name: Khadija Bassiouni*

**Step 1: Setting up the Environment:**

In [2]:
# importing the core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

In [1]:
# Mounting Drive in Colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Load Only Required Columns to Optimize Memory( due to RAM crashing)
import pandas as pd

# Define the exact columns needed for Part 1 (MACD/RSI) and Part 3 (ML Features)
needed_columns = [
    'date', 'close',
    'daily_return', 'close_lag_1', 'ma_5', 'ma_20', 'volatility_20', 'daily_range_pct'
]

# Load only these columns and optimize data types to save RAM
df = pd.read_csv(
    '/content/drive/MyDrive/stock_prepared.csv',
    usecols=lambda col: col in needed_columns,
    dtype={
        'close': 'float32',
        'daily_return': 'float32',
        'close_lag_1': 'float32',
        'ma_5': 'float32',
        'ma_20': 'float32',
        'volatility_20': 'float32',
        'daily_range_pct': 'float32'
    }
)

# Preview the dataframe to verify it loaded successfully
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20860334 entries, 0 to 20860333
Data columns (total 7 columns):
 #   Column         Dtype  
---  ------         -----  
 0   date           object 
 1   close          float32
 2   daily_return   float32
 3   close_lag_1    float32
 4   ma_5           float32
 5   ma_20          float32
 6   volatility_20  float32
dtypes: float32(6), object(1)
memory usage: 636.6+ MB
None


,date,close,daily_return,close_lag_1,ma_5,ma_20,volatility_20
0,1999-12-17,32.859444,-0.027778,33.798283,31.956366,31.209764,0.051155
1,1999-12-20,33.530045,0.020408,32.859444,32.153076,31.442240,0.047086
2,1999-12-21,33.351215,-0.005333,33.530045,32.671673,31.536123,0.043080
3,1999-12-22,34.021816,0.020107,33.351215,33.512161,31.806599,0.036956
4,1999-12-23,35.586552,0.045992,34.021816,33.869816,32.117310,0.037681


**Part 1 — Create Technical Indicators:**

In [4]:
# Calculating MACD
#  12-day and 26-day EMAs of close price
ema_12 = df['close'].ewm(span=12, adjust=False).mean()
ema_26 = df['close'].ewm(span=26, adjust=False).mean()

#  MACD line
macd_line = ema_12 - ema_26

# Signal Line (9-day EMA of MACD)
signal_line = macd_line.ewm(span=9, adjust=False).mean()

df['macd'] = macd_line
df['signal_line'] = signal_line

# MACD Rule: Buy when MACD crosses above signal line, Sell when crosses below, else Hold
df['macd_signal'] = 'Hold'
# Crosses above: yesterday MACD < Signal and today MACD > Signal
buy_condition_macd = (df['macd'].shift(1) < df['signal_line'].shift(1)) & (df['macd'] > df['signal_line'])
sell_condition_macd = (df['macd'].shift(1) > df['signal_line'].shift(1)) & (df['macd'] < df['signal_line'])

df.loc[buy_condition_macd, 'macd_signal'] = 'Buy'
df.loc[sell_condition_macd, 'macd_signal'] = 'Sell'

In [5]:
# Calculate RSI
delta = df['close'].diff()
gain = delta.clip(lower=0)
loss = -1 * delta.clip(upper=0)

# Using exponential moving average for gain and loss (standard 14-period Wilder's smoothing equivalent)
avg_gain = gain.ewm(com=13, adjust=False).mean()
avg_loss = loss.ewm(com=13, adjust=False).mean()

rs = avg_gain / avg_loss
df['rsi'] = 100 - (100 / (1 + rs))

# RSI Rule: Buy if < 30, Sell if > 70, Hold otherwise
df['rsi_signal'] = 'Hold'
df.loc[df['rsi'] < 30, 'rsi_signal'] = 'Buy'
df.loc[df['rsi'] > 70, 'rsi_signal'] = 'Sell'

In [6]:
#  Combined Trading Signal
# Both MACD and RSI must agree for Buy/Sell; otherwise, Hold.
df['trading_signal'] = 'Hold'
df.loc[(df['macd_signal'] == 'Buy') & (df['rsi_signal'] == 'Buy'), 'trading_signal'] = 'Buy'
df.loc[(df['macd_signal'] == 'Sell') & (df['rsi_signal'] == 'Sell'), 'trading_signal'] = 'Sell'

print(df['trading_signal'].value_counts())

trading_signal
Hold    20838968
Sell       10873
Buy        10493
Name: count, dtype: int64


**Part 2 — Check for Class Imbalance:**

In [7]:
signal_counts = df['trading_signal'].value_counts()
signal_percentages = df['trading_signal'].value_counts(normalize=True) * 100

imbalance_df = pd.DataFrame({
    'Count': signal_counts,
    'Percentage (%)': signal_percentages
})
print(imbalance_df)

                   Count  Percentage (%)
trading_signal                          
Hold            20838968       99.897576
Sell               10873        0.052123
Buy                10493        0.050301


**Part 3 — Prepare the Data for Machine Learning:**

In [8]:
# 1. Define feature columns (using the exact columns present in your DataFrame) and target
features = ['daily_return', 'close_lag_1', 'ma_5', 'ma_20', 'volatility_20']
target = 'trading_signal'

# Drop rows with NaN values resulting from rolling indicators/lags
model_df = df[features + [target]].dropna().copy()

# 2. Map text labels to numerical labels (Buy: 0, Hold: 1, Sell: 2)[cite: 1]
label_mapping = {'Buy': 0, 'Hold': 1, 'Sell': 2}
model_df['target_num'] = model_df[target].map(label_mapping)

X = model_df[features]
y = model_df['target_num']

# 3. Time-based Split (Earlier dates for training, later dates for test - no shuffling)
split_index = int(len(model_df) * 0.8)

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

# 4. Standardize features (Fit scaler ONLY on training data to prevent leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



# X_train_scaled, y_train = X_train_scaled[:50000], y_train.iloc[:50000]

print(f"Training set shape: {X_train_scaled.shape}")
print(f"Test set shape: {X_test_scaled.shape}")

Training set shape: (16688267, 5)
Test set shape: (4172067, 5)


**Part 5 — Build Machine Learning Models:**

In [9]:
# taking a representative sample of the training data
sample_size = min(50000, len(X_train_scaled))

# To maintain time-order context, we take the most recent 50k of the training set
#  using a random sample with a fixed seed for reproducibility:
np.random.seed(42)
train_indices = np.random.choice(len(X_train_scaled), size=sample_size, replace=False)

X_train_sample = X_train_scaled[train_indices]
y_train_sample = y_train.iloc[train_indices]

print(f"Using a training sample of {sample_size:,} rows for fast execution.")

# RUNNING MODELS WITH THE SAMPLED DATA
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1), # n_jobs=-1 uses all CPU cores
    'SVM': SVC(random_state=42)
}

evaluation_results = []

for name, model in models.items():
    print(f"\nTraining {name}...")

    # Training model on the sample
    model.fit(X_train_sample, y_train_sample)

    # Predicting on the test set
    y_pred = model.predict(X_test_scaled)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average=None, zero_division=0)

    evaluation_results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': precision,
        'Recall': recall
    })

    print(f"--- {name} Results ---")
    print(f"Accuracy: {acc:.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Buy', 'Hold', 'Sell'], zero_division=0))

Using a training sample of 50,000 rows for fast execution.

Training Logistic Regression...
--- Logistic Regression Results ---
Accuracy: 0.9990
Confusion Matrix:
[[      0    2125       0]
 [      0 4167709       0]
 [      0    2233       0]]

Classification Report:
              precision    recall  f1-score   support

         Buy       0.00      0.00      0.00      2125
        Hold       1.00      1.00      1.00   4167709
        Sell       0.00      0.00      0.00      2233

    accuracy                           1.00   4172067
   macro avg       0.33      0.33      0.33   4172067
weighted avg       1.00      1.00      1.00   4172067


Training Random Forest...
--- Random Forest Results ---
Accuracy: 0.9990
Confusion Matrix:
[[      0    2125       0]
 [      0 4167709       0]
 [      0    2233       0]]

Classification Report:
              precision    recall  f1-score   support

         Buy       0.00      0.00      0.00      2125
        Hold       1.00      1.00      1.00

**Part 7 — Improve One Model:**

In [10]:
# 7. Train Improved Random Forest with Balanced Class Weights
improved_rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1  # Uses all available CPU cores for speed
)

print("Training model...")
improved_rf.fit(X_train_sample, y_train_sample)


Training model...


RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=42)

In [11]:

#  Predict on Test Set and Evaluate
y_pred_imp = improved_rf.predict(X_test_scaled)

print("\n" + "="*40)
print("--- IMPROVED RANDOM FOREST RESULTS ---")
print("="*40)
print(f"Accuracy: {accuracy_score(y_test, y_pred_imp):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_imp))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_imp, target_names=['Buy', 'Hold', 'Sell'], zero_division=0))


--- IMPROVED RANDOM FOREST RESULTS ---
Accuracy: 0.9990

Confusion Matrix:
[[      0    2125       0]
 [      0 4167709       0]
 [      0    2233       0]]

Classification Report:
              precision    recall  f1-score   support

         Buy       0.00      0.00      0.00      2125
        Hold       1.00      1.00      1.00   4167709
        Sell       0.00      0.00      0.00      2233

    accuracy                           1.00   4172067
   macro avg       0.33      0.33      0.33   4172067
weighted avg       1.00      1.00      1.00   4172067

